# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring (regression-style, not classification).**

The output is not a yes/no label but a continuous score per page like how far its 
observed CTR falls below what's typical for pages at its own position tier. 
Pages are then rankable by this score for editorial review. This is deliberately 
*not* classification: a hard "underperforming / not" cutoff would throw away the 
magnitude of the gap, and magnitude is exactly what a reviewer needs to triage 
20 candidates down to the 5 worth doing this week.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy: `ctr_gap` = tier_avg_ctr − observed ctr.**

This is a proxy, not a ground-truth label and there's no dataset that says "this page's 
title is objectively weak." `ctr_gap` stands in for "this page is under-monetizing its 
ranking position," using the tier average as the expected baseline. A positive gap 
means the page earns fewer clicks than peers at the same position, which is a testable 
review trigger, not proof of a bad title.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os
os.chdir('./../../')

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")
visible = df[(df["impressions_90d"] >= 100) & (df["position_tier"].notna())].copy()

visible["tier_avg_ctr"] = visible.groupby("position_tier")["ctr"].transform("mean")
visible["ctr_gap"] = visible["tier_avg_ctr"] - visible["ctr"]   # positive = underperforming
visible[["ctr", "position_tier", "tier_avg_ctr", "ctr_gap"]].head()

,ctr,position_tier,tier_avg_ctr,ctr_gap
0,0.76,striking,0.255782,-0.504218
1,0.05,page_3_5,0.142359,0.092359
2,0.09,page_3_5,0.142359,0.052359
3,0.49,page_1,0.354760,-0.135240
4,0.13,page_3_5,0.142359,0.012359


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@K on the ranked-by-`ctr_gap` list, not accuracy or R².**

The action is "an editor reviews the top N flagged pages this week," so what matters 
is whether the top of the ranking is genuinely worth reviewing and not how well the score 
fits every page in the dataset. R² would reward fitting noise in low-impression pages 
that no one will ever act on. Precision@K (e.g., @50) asks the only question that 
matters operationally: of the top 50 pages by ctr_gap, how many are real, checkable 
underperformers (large gap + enough impression volume to trust the CTR estimate)?

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (content unit), restricted to pages with at least 100 impressions 
over the trailing 90 days and below that, CTR is too noisy to compare meaningfully 
(a page with 12 impressions and 1 click has a "CTR" of 8.3% that means almost nothing).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Rows (pages) after impressions_90d >= 100 filter: {len(visible):,}")
print(f"Columns: {visible.shape[1]}")
visible[["ctr", "avg_position", "position_tier", "tier_avg_ctr", "ctr_gap"]].sort_values(
    "ctr_gap", ascending=False
).head(10)

Rows (pages) after impressions_90d >= 100 filter: 22,006
Columns: 46


,ctr,avg_position,position_tier,tier_avg_ctr,ctr_gap
8939,0.0,7.2,page_1,0.35476,0.35476
29977,0.0,8.2,page_1,0.35476,0.35476
29983,0.0,6.8,page_1,0.35476,0.35476
8873,0.0,5.2,page_1,0.35476,0.35476
8915,0.0,4.9,page_1,0.35476,0.35476
33,0.0,6.8,page_1,0.35476,0.35476
2282,0.0,5.0,page_1,0.35476,0.35476
24588,0.0,6.7,page_1,0.35476,0.35476
11665,0.0,6.2,page_1,0.35476,0.35476
11673,0.0,6.8,page_1,0.35476,0.35476


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule: e.g. "flag any page with CTR < 0.20" and fails here because it ignores 
position tier entirely: 0.20 CTR is *excellent* for a `deep` page and *alarming* for 
a `page_1` page. A single global threshold either misses every real page_1 problem 
or flags thousands of totally normal deep-tier pages. 

The tier-relative approach already beats that naive rule, but it's still a fixed rule 
(subtract the tier mean). What actually earns the "ML" label going forward is that 
this framing generalizes: instead of one hand-picked baseline (tier mean), a model 
can learn expected CTR from *multiple* interacting signals at once, i.e. position, tier, 
query count, seasonality, content age, the way the reference pipeline's model beat 
its own hand-rule baseline by ~3x on Precision@50. A fixed single-variable rule can't 
weigh five signals against each other; a model can.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.